# Part 1: Supervised Learning - Pneumonia Classification

This section of the CVRIE project focuses on preparing a medical dataset of chest X-rays and training machine learning models. The goal is to classify patient records into three distinct categories: Normal (Healthy), Bacteria (Bacterial Pneumonia), or Virus (Viral Pneumonia). 

The following pipeline includes data formatting, mathematical normalization, model training, and performance comparison.

## 1. Library Imports

Before manipulating the dataset, we must import the necessary Python libraries. Here is the utility of each module used in this notebook:

* **`os`**: Allows us to interact with the operating system, navigate through folders, and read file paths.
* **`numpy (np)`**: The core library for scientific computing. It is essential for handling large multi-dimensional arrays (our matrices of pixels) and performing fast mathematical operations.
* **`PIL (Python Imaging Library)`**: Used to open the original images, resize them, and manipulate their color channels.
* **`matplotlib.pyplot (plt)`**: A plotting library used to display images, generate charts, and visualize our model's confusion matrices.
* **`sklearn`**: The Machine Learning library. We import our algorithms (`RandomForestClassifier`, `KNeighborsClassifier`) and our evaluation metrics (`accuracy_score`, `classification_report`, etc.) from here.
* **`joblib`**: The lib that we will use for export the final modal.

In [ ]:
import os
import sys
import numpy as np
from PIL import Image
import joblib
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_auc_score, roc_curve, auc
from sklearn.preprocessing import label_binarize

## 2. Dataset Formatting and Classification


Medical images often come in various shapes, sizes, and color formats. To feed them to a Machine Learning model, they must be strictly uniform. In this section, we read the raw images and save a clean, standardized version to a new directory.

To format the dataset we will:
1. **Resize** all images to `256x256` and change color palette to grayscale
2. **Classify** images into an organized directory structure:
   - `dataset/train/normal` — normal images (train)
   - `dataset/train/bacteria` — bacterial pneumonia (train)
   - `dataset/train/virus` — viral pneumonia (train)
   - `dataset/test/normal` — normal images (test)
   - `dataset/test/bacteria` — bacterial pneumonia (test)
   - `dataset/test/virus` — viral pneumonia (test)
3. **Count** images before and after processing to ensure nothing is lost

## `resize_image(image_path, output_path, dim1, dim2)`

Resizes an image to dimensions `dim1 x dim2` and saves it to `output_path`.
It also convert images to 1 canal (only Grey) with the line `img = img.convert('L')`.
after that the dataset is fully uniformised.

**Parameters:**
- `image_path`: path to the source image
- `output_path`: destination path
- `dim1`, `dim2`: target dimensions

**Key lines of code explained:**
* `Image.open()`: Loads the raw image file from the disk.
* `img.convert('L')`: Converts the image strictly to grayscale. Since some X-rays are saved in RGB and others in grayscale, forcing them to 'L' prevents severe dimension mismatch errors during the array creation.
* `img.resize((dim1, dim2))`: Reduces the image size to dim1 x dim2 pixels. This reduces the number of features per image, preventing memory overload and speeding up the model training.

**Returns:** the path of the saved image

In [ ]:
def resize_image(image_path, output_path, dim1, dim2):
    print("Resizing image:", image_path)
    print("Output path:", output_path)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    img = Image.open(image_path)
    img = img.convert('L')
    img = img.resize((dim1, dim2))
    img.save(output_path)
    return output_path

## `classify_image(filename, source_subdir)`

Determines the destination subdirectory for an image based on its filename and source directory.

**Classification logic:**
- If the image comes from `NORMAL` → `normal` folder
- If the image comes from `PNEUMONIA` and contains **"bacteria"** in its name → `bacteria` folder
- If the image comes from `PNEUMONIA` and contains **"virus"** in its name → `virus` folder
- Otherwise → `None` (image skipped)

**Returns:** the destination subdirectory name (`"normal"`, `"bacteria"`, `"virus"`) or `None`

In [ ]:
def classify_image(filename, source_subdir):
    """Determines the destination folder based on the filename and its source directory."""
    name_lower = filename.lower()
    if source_subdir == "NORMAL":
        return "normal"
    elif source_subdir == "PNEUMONIA":
        if "bacteria" in name_lower:
            return "bacteria"
        elif "virus" in name_lower:
            return "virus"
    return None

## `count_images(filepath)`

Recursively counts the number of images (`.png`, `.jpg`, `.jpeg`) in a given directory.

**Parameter:** `filepath` — path of the directory to scan

**Returns:** the total number of images found

In [ ]:
def count_images(filepath):
    """Compte le nombre d'images dans un dossier récursivement."""
    count = 0
    for root, dirs, filenames in os.walk(filepath):
        for filename in filenames:
            if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                count += 1
    return count

## `format_images(filepath, output_base)`

Main processing function. Iterates through the `train/` and `test/` directories of the source dataset and for each image:
1. Determines its label via `classify_image()`
2. Resizes it to **256x256** via `resize_image()`
3. Saves it to the appropriate destination folder

**Parameters:**
- `filepath`: path to the source dataset (containing `train/` and `test/`)
- `output_base`: root output path (e.g., `./dataset`)

**Returns:** a `np.array` containing the paths of all processed images

In [ ]:
def format_images(filepath, output_base):
    images = []

    if not os.path.exists(filepath):
        sys.stderr.write(f"Error: Directory '{filepath}' does not exist.\n")
        sys.exit(84)

    for split in ["train", "test"]:
        split_path = os.path.join(filepath, split)
        if not os.path.exists(split_path):
            print(f"Directory '{split}' not found")
            continue

        for category in ["NORMAL", "PNEUMONIA"]:
            category_path = os.path.join(split_path, category)
            if not os.path.exists(category_path):
                print(f"Directory '{split}/{category}' not found")
                continue

            for filename in os.listdir(category_path):
                if not filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                    continue
                dest_subdir = classify_image(filename, category)

                if dest_subdir is None:
                    print(f"Image skipped: {filename}")
                    continue

                source_path = os.path.join(category_path, filename)
                output_path = os.path.join(output_base, split, dest_subdir, filename)

                try:
                    result = resize_image(source_path, output_path, 256, 256)
                    images.append(result)
                except Exception as e:
                    sys.stderr.write(f"Error processing image {filename}: {e}\n")
                    sys.exit(84)

    return np.array(images) if images else np.array([])

## `main()`

Entry point of the script, runs in this order:
1. **Pre-processing count** — displays the number of images per source directory (`train/NORMAL`, `train/PNEUMONIA`, `test/NORMAL`, `test/PNEUMONIA` and total)
2. **Processing** — calls `format_images` to resize and classify all images
3. **Post-processing count** — displays the number of images per destination directory (`train/normal`, `train/bacteria`, etc.) to verify all images were processed

In [ ]:
def main():
    dataset_folder = os.path.abspath('./chest_xray')
    output_folder = os.path.abspath('./dataset')

    print("IMAGE COUNT BEFORE PROCESSING")
    total_before = 0
    for split in ["train", "test"]:
        split_path = os.path.join(dataset_folder, split)
        if os.path.exists(split_path):
            for category in ["NORMAL", "PNEUMONIA"]:
                cat_path = os.path.join(split_path, category)
                if os.path.exists(cat_path):
                    n = count_images(cat_path)
                    total_before += n
                    print(f"  {split}/{category}: {n} images")
    print(f"  TOTAL before: {total_before} images")

    formatted_images = format_images(dataset_folder, output_folder)
    print(f"\nNumber of formatted images: {len(formatted_images)}")

    print("IMAGE COUNT AFTER PROCESSING")
    total_after = 0
    for split in ["train", "test"]:
        split_path = os.path.join(output_folder, split)
        if os.path.exists(split_path):
            for category in ["normal", "bacteria", "virus"]:
                cat_path = os.path.join(split_path, category)
                if os.path.exists(cat_path):
                    n = count_images(cat_path)
                    total_after += n
                    print(f"  {split}/{category}: {n} images")
    print(f"  TOTAL after: {total_after} images")

if __name__ == "__main__":
    main()

## `plot_train_test_distribution(train_path, test_path)`

Visualizes the distribution of classes (Normal, Bacteria, Virus) in both the training and test datasets.
This step is crucial to check for data imbalances that might affect the model's performance.

**Parameters:**
- `train_path`: path to the training dataset directory
- `test_path`: path to the test dataset directory

**Key lines of code explained:**
* `plt.subplots()`: Creates a figure with two subplots side-by-side (one for train, one for test).
* `ax.bar()`: Plots the bar charts using the computed counts.

**Outputs:** A matplotlib figure showing the distribution of X-rays per class.

In [ ]:
def plot_train_test_distribution(train_path, test_path):
    labels = ['normal', 'bacteria', 'virus']
    
    train_counts = [len(os.listdir(os.path.join(train_path, l))) if os.path.exists(os.path.join(train_path, l)) else 0 for l in labels]
    test_counts = [len(os.listdir(os.path.join(test_path, l))) if os.path.exists(os.path.join(test_path, l)) else 0 for l in labels]
            
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors = ['#2ca02c', '#d62728', '#1f77b4']
    
    bars_train = axes[0].bar(labels, train_counts, color=colors)
    axes[0].set_title('Training Dataset Class Distribution', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Number of X-Rays')
    
    for bar in bars_train:
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, int(bar.get_height()), ha='center', fontweight='bold')
    bars_test = axes[1].bar(labels, test_counts, color=colors)
    axes[1].set_title('Test Dataset Class Distribution', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('Number of X-Rays')
    for bar in bars_test:
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, int(bar.get_height()), ha='center', fontweight='bold')
    plt.tight_layout()
    plt.show()
plot_train_test_distribution('dataset/train', 'dataset/test')

## `load_data(base_path)`

Flattens the dataset images into matrices and loads the data to train the model. This normalization step converts 2D image representations into 1D arrays which classical Machine Learning algorithms (like Random Forest or KNN) can understand.

**Parameters:**
- `base_path`: path to the dataset split (e.g., `dataset/train` or `dataset/test`)

**Key lines of code explained:**
* `img.max() > 1.0; img / 255.0`: Normalizes the pixel values exactly between 0 and 1, facilitating gradient convergence and distance calculations.
* `img.flatten()`: Flattens the 2D image grid (width x height) into a single continuous 1D array of features.

**Returns:** A tuple `(X, y)` containing the features (flattened images) and labels.

In [ ]:
def load_data(base_path, half_normal=False):
    label_map = {'normal': 0, 'bacteria': 1, 'virus': 2}   
    X = []
    y = []
    for category, label_value in label_map.items():
        category_path = os.path.join(base_path, category)
        if not os.path.exists(category_path):
            continue
        
        file_names = os.listdir(category_path)
        if category == 'normal' and half_normal:
            file_names = file_names[:len(file_names) // 2]
            
        for file_name in file_names:
            file_path = os.path.join(category_path, file_name)
            try:
                img = plt.imread(file_path)
                if img.max() > 1.0:
                    img = img / 255.0
                img_flattened = img.flatten()
                X.append(img_flattened)
                y.append(label_value)
            except Exception as e:
                print(f"Error with image {file_path}: {e}")
    return np.array(X), np.array(y)

print("Loading training data...")
X_train, y_train = load_data('dataset/train')
print("Loading test data...")
X_test, y_test = load_data('dataset/test', half_normal=True)
print(f"Training data dimensions: {X_train.shape}")
print(f"Test data dimensions: {X_test.shape}")

## 3. Model Training and Evaluation (Random Forest)

We will test a **Random Forest Algorithm** on our flattened data to build our image classification model. 
Random Forest operates by constructing a multitude of decision trees at training time and outputting the class that is the mode of the classes of the individual trees, which makes it generally robust against overfitting.

**Steps:**
- `RandomForestClassifier(n_estimators=100)`: Initialize with 100 trees
- `rf_model.fit(X_train, y_train)`: Train the model
- `rf_model.predict(X_test)`: Make predictions
- Metric evaluation using Accuracy, Precision, Recall, and F1-Score
- Generating a Confusion Matrix down to the specific classes.

In [ ]:
print("Initializing Random Forest model...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
print("Training in progress...")
rf_model.fit(X_train, y_train)
print("Training completed!")
print("Predicting on the test set...")
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nOverall accuracy: {accuracy * 100:.2f}%")
print("\nDetailed classification report:")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Bacteria', 'Virus']))
cm_rf = confusion_matrix(y_test, y_pred)
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=['Normal', 'Bacteria', 'Virus'])
disp_rf.plot(cmap=plt.cm.Blues)
plt.title("Random Forest - Confusion Matrix")
plt.show()

## 4. Model Training and Evaluation (K-Nearest Neighbors - KNN)

We will test the **k-Nearest Neighbors (kNN) Algorithm** to evaluate and compare against Random Forest. 
KNN evaluates the k closest training examples in the multi-dimensional feature space to predict the output class.

**Steps:**
- `KNeighborsClassifier(n_neighbors=3)`: Initialize the model to classify a point based on its 3 nearest neighbors.
- `knn_model.fit(X_train, y_train)`: Train the model on flattened image datasets.
- Evaluation on test data via Classification Report and Confusion Matrix visualization.

In [ ]:
print("Initializing KNN model...")
knn_model = KNeighborsClassifier(n_neighbors=3, n_jobs=-1)

print("KNN training in progress...")
knn_model.fit(X_train, y_train)
print("Training completed!")

print("Predicting on the test set...")
y_pred_knn = knn_model.predict(X_test)
accuracy_knn = accuracy_score(y_test, y_pred_knn)

print(f"\nOverall KNN accuracy: {accuracy_knn * 100:.2f}%")

print("\nDetailed classification report (KNN):")
print(classification_report(y_test, y_pred_knn, target_names=['Normal', 'Bacteria', 'Virus']))

cm_knn = confusion_matrix(y_test, y_pred_knn)
disp_knn = ConfusionMatrixDisplay(confusion_matrix=cm_knn, display_labels=['Normal', 'Bacteria', 'Virus'])
disp_knn.plot(cmap=plt.cm.Oranges)
plt.title("KNN - Confusion Matrix")
plt.show()

## 5. Model Comparison

After training the **Random Forest** and **K-Nearest Neighbors (KNN)** algorithms on the extracted images datasets, we can visualize and compare their performance mathematically relative to our classification task. The primary performance metric shown below is overall classification `Accuracy (%)`.

**Visual Insight:**
*   A bar chart visualizing and comparing the test metrics to see which model best applies to this particular 3D X-Ray Dataset.

In [ ]:
models = ['Random Forest', 'KNN']
accuracies = [accuracy * 100, accuracy_knn * 100] 

plt.figure(figsize=(6, 4))
bars = plt.bar(models, accuracies, color=['#1f77b4', '#ff7f0e'])
plt.title('Algorithm Accuracy Comparison', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy (%)')
plt.ylim(0, 100)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 2, f"{yval:.2f}%", ha='center', fontweight='bold')
    
plt.show()

## 6. Optimization of the Chosen Model

Given the outcome, the **Random Forest** typically outperforms K-Nearest Neighbors in this task. We now focus on determining the best model parameters by systematically testing hyper-parameters combos.

We use **`GridSearchCV`** from `sklearn.model_selection` to perform an exhaustive search across specified parameter values for the Random Forest Classifier.

`GridSearchCV` tests parameters like:
- `n_estimators`: The number of trees in the forest.
- `max_depth`: The maximum depth of the tree to prevent overfitting.
- `min_samples_split`: The minimum number of samples required to split an internal node.
- `class_weight`: Handles class imbalances.

**(Note: This code block might be computationally intensive and takes several hours, it's commented out in practice to prevent blocking continuous execution).**

In [ ]:
# from sklearn.model_selection import GridSearchCV

# print("Starting TOTAL optimization of Random Forest")

# param_grid = {
#     'n_estimators': [50, 100, 200, 300, 400, 500],
#     'max_depth': [None, 10, 20, 30, 40, 50],
#     'min_samples_split': [2, 5, 10, 15, 20],
#     'class_weight': ['balanced', None, 'balanced_subsample']
# }

# base_rf = RandomForestClassifier(random_state=42, n_jobs=-1)

# rf_grid_search = GridSearchCV(
#     estimator=base_rf, 
#     param_grid=param_grid,
#     cv=3,      
#     verbose=2, 
#     n_jobs=1
# )

# print("Searching for best parameters in progress...")
# print("WARNING: 540 combinations x 3 folds = 1620 models to train sequentially. This may take several hours.")
# rf_grid_search.fit(X_train, y_train)

# best_rf_model = rf_grid_search.best_estimator_

# print(f"\nBest hyperparameters found among all combinations: {rf_grid_search.best_params_}")

# print("\nPredicting with the optimized model...")
# y_pred_best = best_rf_model.predict(X_test)
# accuracy_best = accuracy_score(y_test, y_pred_best)

# print(f"\nNew overall accuracy: {accuracy_best * 100:.2f}%")
# print("\nNew classification report:")
# print(classification_report(y_test, y_pred_best, target_names=['Normal', 'Bacteria', 'Virus']))

# cm_best = confusion_matrix(y_test, y_pred_best)
# disp_best = ConfusionMatrixDisplay(confusion_matrix=cm_best, display_labels=['Normal', 'Bacteria', 'Virus'])
# disp_best.plot(cmap=plt.cm.Greens)
# plt.title("Ultimate Optimized Random Forest - Confusion Matrix")
# plt.show()

### 7. Post-Optimization Training (Hyperparameter Validation)

**Result of `GridSearchCV` Optimization:**
After conducting the search, the best combination of hyperparameters was determined to be:
*   `class_weight`: **'balanced'**
*   `max_depth`: **20**
*   `min_samples_split`: **5**
*   `n_estimators`: **500**

**Expected Outcome based on previously conducted training runs:**
- Overall Accuracy: ~`76.12%`

**Classification Report Summary:**
```text
              precision    recall  f1-score   support

      Normal       0.93      0.61      0.74       234
    Bacteria       0.73      0.97      0.83       242
       Virus       0.65      0.67      0.66       148

    accuracy                           0.76       624
```

We now instantly re-train our Random Forest Model using these optimal hyperparameters instead of waiting for `GridSearchCV` to run again.

In [ ]:
print("Training Optimized Random Forest")

ultimate_rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=20,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

print("Training in progress...")
ultimate_rf_model.fit(X_train, y_train)

y_pred_ultimate = ultimate_rf_model.predict(X_test)
accuracy_ultimate = accuracy_score(y_test, y_pred_ultimate)

print(f"\nOverall accuracy: {accuracy_ultimate * 100:.2f}%")
print("\nClassification report:")
print(classification_report(y_test, y_pred_ultimate, target_names=['Normal', 'Bacteria', 'Virus']))
y_prob_ultimate = ultimate_rf_model.predict_proba(X_test)
roc_auc = roc_auc_score(y_test, y_prob_ultimate, multi_class='ovr')
print(f"\nGlobal ROC AUC Score: {roc_auc:.4f}")
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
target_names = ['Normal', 'Bacteria', 'Virus']
colors = ['#2ca02c', '#d62728', '#1f77b4']

plt.figure(figsize=(10, 7))

for i, color in zip(range(3), colors):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob_ultimate[:, i])
    roc_auc_class = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2,
             label=f'ROC {target_names[i]} (Area = {roc_auc_class:.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Chance (Area = 0.50)') 
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve - Optimized Random Forest', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=11)
plt.grid(alpha=0.3)
plt.show()

cm_ultimate = confusion_matrix(y_test, y_pred_ultimate)
disp_ultimate = ConfusionMatrixDisplay(confusion_matrix=cm_ultimate, display_labels=['Normal', 'Bacteria', 'Virus'])
disp_ultimate.plot(cmap=plt.cm.Greens)
plt.title("Ultimate Optimized Random Forest - Confusion Matrix")
plt.show()

## 8. Advanced Experiment: PCA + Neural Network (MLP) Analysis

To push the limits of our authorized Scikit-learn tools, we attempted a dimensional reduction using **PCA (Principal Component Analysis)**, keeping 95% of the variance. 
This successfully compresses our data from tens of thousands of features (pixels) down to highly abstract components. We then fed this compressed data to a **Multi-Layer Perceptron (`MLPClassifier`)**, a basic neural network.

**The Insight:**
Contrary to the assumption that Neural Networks always perform better, this experiment yielded a distinctly lower accuracy than our Optimized Random Forest. 
1. **Spatial Awareness:** Standard MLPs process 1D arrays and cannot natively understand 2D spatial relationships (like a localized lung anomaly shape). Advanced image recognition relies on Convolutional Neural Networks (CNNs), which are beyond the scope of Scikit-learn alone.
2. **Algorithmic Robustness:** The Random Forest, being an ensemble method (voting amongst 500 trees in this instance), remains naturally more resilient to the noise and high-dimensionality of flattened raw pixels than a standard single MLP.

**Final Verdict for CVRIE V1:**
The **Optimized Random Forest** represents our absolute best model for this supervised classification task.

In [ ]:
print("1. Dimensionality reduction with PCA")
pca = PCA(n_components=0.95, random_state=42)

print("Image compression in progress...")
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print(f"Original dimensions: {X_train.shape[1]} pixels.")
print(f"New dimensions after PCA: {X_train_pca.shape[1]} highly relevant components!")

print("\n2. Neural Network (MLP) Training")
mlp_model = MLPClassifier(
    hidden_layer_sizes=(100, 50), 
    activation='relu', 
    solver='adam', 
    max_iter=500,
    random_state=42
)

print("Neural network training in progress...")
mlp_model.fit(X_train_pca, y_train)

print("\n3. Evaluation of the new model")
y_pred_mlp = mlp_model.predict(X_test_pca)
accuracy_mlp = accuracy_score(y_test, y_pred_mlp)

print(f"\nOverall accuracy PCA + MLP: {accuracy_mlp * 100:.2f}%")
print("\nClassification report:")
print(classification_report(y_test, y_pred_mlp, target_names=['Normal', 'Bacteria', 'Virus']))

cm_mlp = confusion_matrix(y_test, y_pred_mlp)
disp_mlp = ConfusionMatrixDisplay(confusion_matrix=cm_mlp, display_labels=['Normal', 'Bacteria', 'Virus'])
disp_mlp.plot(cmap=plt.cm.Purples)
plt.title("PCA + Neural Network - Confusion Matrix")
plt.show()

## 9. Model Serialization (Export)

**Purpose:** 
Training a model, especially an optimized Random Forest with 500 estimators, is computationally and temporally expensive. To deploy CVRIE V1 in a real hospital environment (or reuse it across other analytical scripts without retraining every single time), we must serialize the model object to the hard drive. 

We use the `joblib` library, which is highly optimized for saving large Scikit-learn models (especially those containing significant nested NumPy arrays). The chosen model object is exported as a `.pkl` (pickle) file to be seamlessly loaded elsewhere.

In [ ]:
model_filename = 'cvrie_rf_model_optimized.pkl'
print("Saving model...")
joblib.dump(ultimate_rf_model, model_filename)
print(f"Model successfully saved as: {model_filename}")